# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

1. `1 1.3333333333333333`. В ряду один пропуск. После замены нулём среднее равно (1 + 0 + 3) / 3.
2. `2`. После удаления пробелов и смены регистра остаются «a» и «b».
3. `3`. Результат удаления дублей не присвоен, исходная таблица сохранила три строки.
4. `1005`. После удаления пробела складываются числа 1000 и 5.
5. `[9, 10]`. Первая дата в сентябре, вторая в октябре.

In [1]:
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

1 1.3333333333333333
2
3
1005
[9, 10]


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [3]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    data = pd.read_csv(path)
    print("Прочитано строк", len(data))
    data["amount"] = pd.to_numeric(data["amount"].str.replace(" ", ""))
    assert data["amount"].notna().all()
    print("После перевода сумм", len(data))
    data["city"] = data["city"].str.strip().str.lower().map(CITY)
    assert data["city"].notna().all()
    print("После очистки городов", len(data))
    data["order_date"] = pd.to_datetime(data["order_date"])
    print("После перевода дат", len(data))
    data = data[data["customer_id"] != "TEST"]
    print("Без TEST", len(data))
    data = data.drop_duplicates().copy()
    assert data["order_id"].is_unique
    print("Без дублей", len(data))
    return data

df = clean("orders.csv")

Прочитано строк 5463
После перевода сумм 5463
После очистки городов 5463
После перевода дат 5463
Без TEST 5460
Без дублей 5280


**Вывод для менеджера.** Для анализа осталось 5 280 реальных заказов. Отменённые входят в это число, поэтому по таблице можно считать долю отмен.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [4]:
paid = df[df["status"] == "оплачен"]

def revenue(data):
    month = data["order_date"].dt.month
    sep = data.loc[month == 9, "amount"].sum()
    octo = data.loc[month == 10, "amount"].sum()
    return sep, octo, (octo / sep - 1) * 100

print("Город | Сентябрь | Октябрь | Изменение")
cities = sorted(df["city"].unique())
for city in ["Все города"] + cities:
    part = paid if city == "Все города" else paid[paid["city"] == city]
    sep, octo, change = revenue(part)
    print(f"{city}: {sep:,.0f} | {octo:,.0f} | {change:+.2f}%")

Город | Сентябрь | Октябрь | Изменение
Все города: 35,243,410 | 34,419,520 | -2.34%
Актобе: 2,576,960 | 2,540,420 | -1.42%
Алматы: 14,501,070 | 14,643,020 | +0.98%
Астана: 9,034,220 | 8,850,530 | -2.03%
Атырау: 923,990 | 969,860 | +4.96%
Караганда: 3,648,520 | 3,371,680 | -7.59%
Шымкент: 4,558,650 | 4,044,010 | -11.29%


**Вывод для менеджера.** Итоги совпадают с уроком 4. Выручка снизилась с 35 243 410 до 34 419 520 (−2,34%). Самое сильное относительное снижение в Шымкенте, примерно на 11,3%. Начать разбор стоит с этого города.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг | Убрано строк | Изменение выручки октября к сентябрю |
|---|---:|---:|
| Расчёт менеджера по оплаченным заказам после перевода сумм в числа | 0 | −22,7% |
| Привели города к одному написанию | 0 | −22,7% |
| Перевели даты в формат даты | 0 | −22,7% |
| Убрали TEST | 3 | −8,2% |
| Убрали дубли | 180 | −2,3% |

**Вывод для менеджера.** Сильнее всего исказили отчёт три тестовых заказа. Они завысили базу сентября и преувеличили падение на 14,5 процентного пункта. При следующем расчёте стоит проверять и тестовые записи, и повторяющиеся заказы.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
missing = df[df["delivery_days"].isna()]
print("Без срока доставки", len(missing))
print(missing["city"].value_counts())
print(missing["status"].value_counts())
days = df.loc[df["city"] == "Шымкент", "delivery_days"]
print("Известные сроки", days.count(), "из", len(days), "| медиана", days.median())
for label, values in [("dropna", days.dropna()), ("медиана", days.fillna(days.median())),
                      ("ноль", days.fillna(0))]:
    print(f"{label}: {(values <= 2).sum()} из {len(values)} = {(values <= 2).mean():.1%}")

Без срока доставки 158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64
Известные сроки 754 из 776 | медиана 5.0
dropna: 225 из 754 = 29.8%
медиана: 225 из 776 = 29.0%
ноль: 247 из 776 = 31.8%


**Вывод для менеджера.** Из 158 пропусков больше всего в Алматы (63), по статусу — у оплаченных заказов (144). Долю доставок за два дня или быстрее стоит указывать среди заказов с известным сроком. В Шымкенте срок неизвестен для 22 заказов. Медиана приписывает им типичный срок, а ноль засчитывает их все как быстрые доставки.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [6]:
data = pd.read_csv("orders.csv")
# Сохраняем результат удаления полных дублей.
data = data.drop_duplicates().copy()
# Учитываем пробелы, регистр и латинские варианты одного названия.
data["city"] = data["city"].str.strip().str.lower().map(CITY)
assert data["city"].notna().all()
print("Городов", data["city"].nunique(), "| строк", len(data))

Городов 6 | строк 5283


**Вывод для менеджера.** После этих двух исправлений отчёт охватывает 6 городов и 5 283 заказа. Для расчёта реальных продаж из них ещё нужно исключить три тестовых.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
counts = df["order_date"].dt.day_name().value_counts()
print(counts)
print(f"Разница относительно максимума: {(1 - counts.min() / counts.max()):.1%}")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Разница относительно максимума: 13.0%


**Вывод для менеджера.** За весь период в понедельник было 684 заказа, на 13,0% меньше, чем в четверг (786). В сентябре и октябре было восемь понедельников и девять четвергов. Средняя нагрузка за один такой день близка (85,5 и 87,3 заказа), поэтому сокращать понедельничную смену на 13% по этим итогам рискованно.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
known = df.dropna(subset=["delivery_days"]).sort_values("order_date").copy()
known["date"] = known["order_date"].dt.normalize()  # календарный день без времени
sh = known[known["city"] == "Шымкент"]
for month in [9, 10]:
    print(month, "месяц, среднее", round(sh.loc[sh["date"].dt.month == month, "delivery_days"].mean(), 2))
last_fast = sh.loc[sh["delivery_days"] < 5, "date"].max()
start = sh.loc[sh["date"] > last_fast, "date"].min()
print("Первая дата", start.date(), "| минимум с этой даты", sh.loc[sh["date"] >= start, "delivery_days"].min())
print("Город | среднее накануне | среднее в день изменения | минимум в этот день")
for city in cities:
    part = known[known["city"] == city]
    before = part.loc[part["date"] == start - pd.Timedelta(days=1), "delivery_days"]
    after = part.loc[part["date"] == start, "delivery_days"]
    print(f"{city}: {before.mean():.1f} | {after.mean():.1f} | {after.min():.0f}")

9 месяц, среднее 2.03
10 месяц, среднее 6.54
Первая дата 2026-10-01 | минимум с этой даты 5.0
Город | среднее накануне | среднее в день изменения | минимум в этот день
Актобе: 1.2 | 2.0 | 1
Алматы: 2.0 | 1.7 | 1
Астана: 1.7 | 1.9 | 1
Атырау: 1.7 | 2.0 | 2
Караганда: 2.1 | 2.2 | 1
Шымкент: 2.3 | 6.2 | 5


**Вывод для менеджера.** С 1 октября все известные сроки в Шымкенте составляют минимум 5 дней. В остальных пяти городах в этот день по-прежнему встречались доставки быстрее пяти дней. В первую очередь стоит проверить, менялись ли с 1 октября местный перевозчик, маршрут отгрузки или порядок регистрации срока.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
def to_amount(s):
    numbers = s.astype("string").str.replace(" ", "").str.replace(",", ".")
    return pd.to_numeric(numbers, errors="coerce").astype(float)

sample = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print(to_amount(sample).tolist())
amount = to_amount(pd.read_csv("orders.csv")["amount"])
print("Пропуски", amount.isna().sum(), "| сумма", amount.sum())
assert amount.notna().all() and amount.sum() == 109883360

[15990.0, 15990.0, 1200.5, nan]
Пропуски 0 | сумма 109883360.0


**Вывод для менеджера.** Все исходные суммы распознаны. Контрольный итог 109 883 360 совпадает с уроком. Он включает тестовые, повторные и отменённые заказы, поэтому для отчёта о продажах нужна выручка из задания 2.